In [37]:
require 'gd'
# this function show the image on the notebook
def display(file_name)
    IRuby.display "<img src='#{file_name}' />", mime: "text/html"
end

:display

In [40]:
# ═════════════════════════════════════════════════════════════════
#  HelixPlot — replicates matplotlib fill_between3d exactly
#
#  Rotation matches matplotlib azim=-60, elev=30 (defaults)
#  Quad winding matches matplotlib fill_between implementation:
#    verts = [p1[i], p1[i+1], p2[i+1], p2[i]]
#  fov/cam_dist computed to fill canvas
# ═════════════════════════════════════════════════════════════════
class HelixPlot

  def initialize(
    width:    600,
    height:   600,
    azim:    -60.0,   # matplotlib azim — rotation around Z
    elev:     30.0,   # matplotlib elev — tilt up
    fov:      3035.0, # computed to fill canvas at these angles
    cam_dist: 20.0,
    cx_off:  -3.0,
    cy_off:   34.0,
    bg:       [252, 252, 252]
  )
    @width    = width
    @height   = height
    @fov      = fov.to_f
    @cam_dist = cam_dist.to_f
    @cx_off   = cx_off.to_f
    @cy_off   = cy_off.to_f
    @bg       = bg

    a = azim * Math::PI / 180.0
    e = elev * Math::PI / 180.0

    # Matplotlib rotation: first azim around Z, then elev around new X
    @ca = Math.cos(a); @sa = Math.sin(a)
    @ce = Math.cos(e); @se = Math.sin(e)
  end

  def render(path = "/work/jupyter-notebooks/helix.png")
    img = new_image

    n     = 50   # same as matplotlib example
    theta = n.times.map { |i| i.to_f / (n-1) * 2 * Math::PI }
    z_arr = n.times.map { |i| i.to_f / (n-1) }

    helix1 = theta.zip(z_arr).map { |t,z| [Math.cos(t),           Math.sin(t),           z] }
    helix2 = theta.zip(z_arr).map { |t,z| [Math.cos(t+Math::PI),  Math.sin(t+Math::PI),  z] }

    # ── Ribbon quads — exact matplotlib winding ───────────────────
    # verts = [p1[i], p1[i+1], p2[i+1], p2[i]]
    patches = []
    (n-1).times do |i|
      pts3d   = [helix1[i], helix1[i+1], helix2[i+1], helix2[i]]
      proj    = pts3d.map { |p| rotate3d(*p) }
      avg_dep = proj.sum { |p| p[2] } / 4.0
      patches << { proj: proj, depth: avg_dep }
    end
    patches.sort_by! { |p| -p[:depth] }

    fill_color = GD::Color.rgb( 70, 130, 200)
    edge_color = GD::Color.rgb( 40,  90, 160)

    patches.each do |patch|
      pts2d = patch[:proj].map { |p| project2d(*p) }
      img.filled_polygon(pts2d, fill_color) rescue nil
      pts2d.each_cons(2) { |a,b| safe_line(img, a, b, edge_color) }
      safe_line(img, pts2d.last, pts2d.first, edge_color)
    end

    # ── Helix curves on top — linewidth=2 simulated with 2 offsets ─
    curve_color = GD::Color.rgb(15, 55, 140)
    [helix1, helix2].each do |helix|
      pts2d = helix.map { |p| project2d(*rotate3d(*p)) }
      pts2d.each_cons(2) do |a, b|
        safe_line(img, a, b, curve_color)
        # simulate linewidth=2
        safe_line(img, [a[0]+1,a[1]], [b[0]+1,b[1]], curve_color)
      end
    end

    # ── Cage ──────────────────────────────────────────────────────
    draw_cage(img)

    img.save(path)
  end

  private

  def draw_cage(img)
    cage_color = GD::Color.rgb(195, 195, 195)
    s = 1.2

    corners = {
      blb: [-s,-s,0], brb: [ s,-s,0], flb: [-s, s,0], frb: [ s, s,0],
      blt: [-s,-s,1], brt: [ s,-s,1], flt: [-s, s,1], frt: [ s, s,1],
    }

    edges = [
      [:blb,:brb],[:brb,:frb],[:frb,:flb],[:flb,:blb],
      [:blt,:brt],[:brt,:frt],[:frt,:flt],[:flt,:blt],
      [:blb,:blt],[:brb,:brt],[:frb,:frt],[:flb,:flt],
    ]

    edges.each do |a_key, b_key|
      a = project2d(*rotate3d(*corners[a_key]))
      b = project2d(*rotate3d(*corners[b_key]))
      safe_line(img, a, b, cage_color)
    end
  end

  # Matplotlib-equivalent rotation:
  # 1. rotate around Z by azim
  # 2. rotate around new X by elev
  def rotate3d(x, y, z)
    # Step 1 — azim around Z
    x1 =  @ca*x + @sa*y
    y1 = -@sa*x + @ca*y
    z1 =  z
    # Step 2 — elev around new X
    x2 =  x1
    y2 =  @ce*y1 - @se*z1
    z2 =  @se*y1 + @ce*z1
    [x2, y2, z2]
  end

  def project2d(x, y, z)
    d  = @cam_dist - z
    d  = 0.1 if d < 0.1
    sx = ( @fov * x / d + @width  / 2.0 + @cx_off).to_i
    sy = (-@fov * y / d + @height / 2.0 + @cy_off).to_i
    [sx, sy]
  end

  def new_image
    img = GD::Image.new(@width, @height)
    img.filled_rectangle(0, 0, @width-1, @height-1, GD::Color.rgb(*@bg))
    img
  end

  def safe_line(img, a, b, color)
    img.line(
      a[0].clamp(0,@width-1),  a[1].clamp(0,@height-1),
      b[0].clamp(0,@width-1),  b[1].clamp(0,@height-1),
      color
    )
  rescue; nil; end
end

:safe_line

In [42]:
# ═════════════════════════════════════════════════════════════════
#  Run
# ═════════════════════════════════════════════════════════════════
name = "helix2.png"

HelixPlot.new(
  width:    600,
  height:   600,
  azim:    -60.0,
  elev:     30.0,
  fov:      3035.0,
  cam_dist: 20.0,
  cx_off:  -3.0,
  cy_off:   34.0
).render("/work/jupyter-notebooks/#{name}")

display(name)

"<img src='helix2.png' />"

Interrupt: 